In [13]:
from embedder import Embedder

In [15]:
# Q1. Embedding a query

embed = Embedder()

q = "How does approximate nearest neighbor search work?"
q_emb = embed.encode(q)
q_emb[0]

np.float64(-0.020582036807885073)

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
for doc in documents:
    if doc['filename']=='02-vector-search/lessons/07-sqlitesearch-vector.md':
        content_07 = doc['content']
        break


In [12]:
content_07_emb = embed.encode(content_07)

In [16]:
content_07_emb.dot(q_emb)

np.float64(0.361070280302606)

In [18]:
# Q3. Chunking and search by hand
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [86]:
from tqdm.auto import tqdm
import numpy as np
texts = [doc["content"] + " " + doc["filename"] for doc in chunks]

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/6 [00:00<?, ?it/s]

In [87]:
scores = X.dot(q_emb)
idx = np.argmax(scores)
chunks[idx]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [88]:
# Q4. Vector search with minsearch
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [89]:
embed = Embedder()

q4 = "What metric do we use to evaluate a search engine?"
q4_emb = embed.encode(q4)

In [90]:
results = vindex.search(q4_emb, num_results=5)
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

In [79]:
#Q5. Text search vs vector search


  0%|          | 0/2 [00:00<?, ?it/s]

In [97]:
q5 = 'How do I store vectors in PostgreSQL?'
q5_emb = embed.encode(q5)
results = vindex.search(q5_emb, num_results=5)

In [98]:
for i in range(len(results)):
    print(results[i]['filename'])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [93]:
from minsearch import Index

index = Index(
    text_fields=["content"])

index.fit(chunks)

In [96]:
search_results = index.search(
    q5,
    num_results=5
)

for i in range(len(search_results)):
    print(search_results[i]['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [102]:
#Q6. Hybrid search
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [103]:
q6 = 'How do I give the model access to tools?'

q6_emb = embed.encode(q6)
vector_results = vindex.search(q6_emb, num_results=5)

text_results = index.search(
    q6,
    num_results=5
)

In [104]:
results = rrf([vector_results, text_results])

In [105]:
for i in range(len(results)):
    print(results[i]['filename'])

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md


In [109]:
print(vector_results[0]['filename'])
print(text_results[0]['filename'])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
